# 🌱 KOOBO — Entraînement du modèle de détection HORS-LIGNE

Entraîne un **modèle léger** (MobileNetV3-Small) de détection de maladies des plantes,
puis l'exporte en **TensorFlow.js** pour tourner **dans le navigateur** (hors-ligne, sans API).
Compatible **Kaggle** et **Google Colab**.

## ✅ À faire AVANT de lancer
**Sur Kaggle :**
1. Panneau de droite → **Settings** → **Accelerator : GPU T4 x2** (ou GPU).
2. **Settings → Internet : On** ← INDISPENSABLE (sinon le téléchargement des datasets échoue).
3. Puis **Run All**.

**Sur Colab :** Exécution → Type d'exécution → **GPU**, puis **Tout exécuter**.

Aucun téléchargement manuel : les datasets sont récupérés automatiquement.
À la fin, **`koobo_web_model.zip`** est créé (Kaggle : onglet **Output** à droite ; Colab : téléchargement auto).


## 1. Installation


In [1]:
# tensorflow & tensorflow_datasets sont déjà présents dans Colab (tfds expose
# bien .load et try_gcs). On NE met PAS à jour tensorflow-datasets ici : un
# --upgrade en cours de session casse l'import (deps etils/array_record).
# On installe seulement le convertisseur TensorFlow.js (utilisé à l'étape 10).
!pip -q install tensorflowjs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
db-dtypes 1.7.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.1

## 2. Imports & configuration


In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import json, os, shutil
print('TensorFlow', tf.__version__)
print('GPU dispo :', tf.config.list_physical_devices('GPU') or 'AUCUN (activez le GPU !)')

IMG = 224           # taille d'entrée du modèle
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE


ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/co

TensorFlow 2.19.0
GPU dispo : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Garde-fou tensorflow_datasets.
# Selon l'environnement, tfds peut se retrouver « cassé » : import à moitié
# rechargé après un pip install, ou masqué par un fichier fantôme
# 'tensorflow_datasets.py' dans le dossier courant → tfds.load disparaît.
# On force ici un import PROPRE du vrai paquet.
import sys, importlib, os
if os.path.isfile('tensorflow_datasets.py'):     # fichier fantôme qui masque le paquet
    os.remove('tensorflow_datasets.py')
    print("Fichier fantôme 'tensorflow_datasets.py' supprimé.")
for m in [k for k in list(sys.modules) if k.startswith('tensorflow_datasets')]:
    del sys.modules[m]                           # purge un éventuel import cassé en cache
importlib.invalidate_caches()
import tensorflow_datasets as tfds               # réimport propre (réassigne le tfds global)
assert hasattr(tfds, 'load'), (
    'tensorflow_datasets mal chargé (pas de .load) — '
    'Menu « Exécution → Redémarrer la session », puis « Tout exécuter ».'
)
print('tensorflow_datasets', tfds.__version__, '— tfds.load OK ✅')

ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/schema.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/core/

AssertionError: tensorflow_datasets mal chargé (pas de .load) — Menu « Exécution → Redémarrer la session », puis « Tout exécuter ».

## 3. Jeux de données (téléchargés automatiquement via le miroir Google Cloud)
On combine **quatre** datasets ouverts via TensorFlow Datasets (aucun téléchargement manuel) :
- **PlantVillage** (~54 000 images, 38 classes) — référence multi-cultures
- **Cassava** (~9 400 images, 5 classes) — manioc, culture clé en Afrique
- **Beans** (~1 300 images, 3 classes) — haricot, images terrain (Ouganda)
- **Citrus Leaves** (~600 images, 4 classes) — agrumes

⚠️ Les URLs sources d'origine de plusieurs de ces datasets sont régulièrement **hors-service**
(Mendeley pour PlantVillage / Citrus, bucket `ibeans` en **403** pour Beans). Le notebook les
charge donc **en priorité depuis le miroir public Google Cloud** (`try_gcs=True`), qui héberge
des versions déjà préparées, avec **repli automatique** sur la source d'origine. Chaque dataset
garde ses propres classes (préfixées par sa source). Split 80 % / 10 % / 10 % chacun, puis fusion.
Si un dataset reste indisponible, il est ignoré et les autres continuent.

In [ ]:
import os
if os.path.exists('tensorflow_datasets.py'):
    os.remove('tensorflow_datasets.py')
    print("Fichier fantôme supprimé !")
else:
    print("Aucun fichier fantôme trouvé.")

In [ ]:
# 4 datasets via TFDS. Les URLs sources d'origine de plusieurs datasets sont
# souvent HS (Mendeley pour PlantVillage/Citrus, 403 sur le bucket ibeans pour Beans).
# → On charge d'abord depuis le MIROIR Google Cloud public (try_gcs=True), qui héberge
#   des versions déjà préparées ; repli sur la source d'origine sinon. Si un dataset reste
#   indisponible, il est ignoré (les autres continuent).
DATASETS = ['plant_village', 'cassava', 'beans', 'citrus_leaves']

CLASSES = []
train_parts, val_parts, test_parts = [], [], []

for name in DATASETS:
    loaded, last_err = None, None
    for use_gcs in (True, False):   # 1) miroir GCS (robuste)  2) source d'origine
        try:
            (d_tr, d_va, d_te), info = tfds.load(
                name, split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
                as_supervised=True, with_info=True, try_gcs=use_gcs)
            loaded = (d_tr, d_va, d_te, info, use_gcs)
            break
        except Exception as e:
            last_err = e
    if loaded is None:
        print(f'⚠️  {name} ignoré : {str(last_err)[:150]}')
        continue
    d_tr, d_va, d_te, info, use_gcs = loaded
    local_names = info.features['label'].names
    o = len(CLASSES)                            # décalage pour des indices globaux uniques
    CLASSES += [f'{name}:{n}' for n in local_names]
    train_parts.append(d_tr.map(lambda x, y, o=o: (x, y + o), AUTOTUNE))
    val_parts.append(d_va.map(lambda x, y, o=o: (x, y + o), AUTOTUNE))
    test_parts.append(d_te.map(lambda x, y, o=o: (x, y + o), AUTOTUNE))
    print(f'✅ {name}: {len(local_names)} classes (source: {"GCS" if use_gcs else "origine"})')

assert train_parts, 'Aucun dataset chargé — vérifiez que Internet est ACTIVÉ.'
NUM = len(CLASSES)
print('TOTAL :', NUM, 'classes')

def concat(parts):
    out = parts[0]
    for p in parts[1:]:
        out = out.concatenate(p)
    return out

raw_train, raw_val, raw_test = concat(train_parts), concat(val_parts), concat(test_parts)

## 4. Pipeline de données + augmentation
L'augmentation est appliquée **dans le pipeline** (pas dans le modèle) → modèle propre, 100 % convertible en TF.js.
On garde les images en **[0, 255]** : MobileNetV3 (`include_preprocessing=True`) fait la normalisation lui-même.


In [ ]:
def prep(img, label):
    img = tf.image.resize(img, (IMG, IMG))
    img = tf.cast(img, tf.float32)   # reste en 0-255
    return img, label

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
])

train = (raw_train.map(prep, AUTOTUNE).shuffle(3000).batch(BATCH)
         .map(lambda x, y: (augment(x, training=True), y), AUTOTUNE).prefetch(AUTOTUNE))
val   = raw_val.map(prep, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
test  = raw_test.map(prep, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)


## 5. Modèle (transfer learning — MobileNetV3-Small)


In [ ]:
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet',
    include_preprocessing=True)   # le modèle normalise lui-même les entrées 0-255
base.trainable = False

inputs = tf.keras.Input((IMG, IMG, 3))           # attend des pixels 0-255
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 6. Entraînement — Phase 1 (base gelée)


In [ ]:
model.fit(train, validation_data=val, epochs=8)


## 7. Entraînement — Phase 2 (fine-tuning, lr faible)


In [ ]:
base.trainable = True
for layer in base.layers[:-40]:   # on n'affine que les dernières couches
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train, validation_data=val, epochs=4)


## 8. Évaluation


In [ ]:
loss, acc = model.evaluate(test)
print('Précision sur le test :', round(acc * 100, 2), '%')


## 9. Sauvegarde du modèle + des classes


In [ ]:
model.save('koobo_disease.keras')
model.save('koobo_model.h5')   # format de secours pour la conversion TF.js
with open('classes.json', 'w', encoding='utf-8') as f:
    json.dump(CLASSES, f, ensure_ascii=False)
print('Modèle (.keras + .h5) et classes.json sauvegardés.')


## 10. Conversion en TensorFlow.js (pour le navigateur)
Produit un dossier `web_model/` : `model.json` + fichiers `.bin` + `classes.json`.


In [ ]:
shutil.rmtree('web_model', ignore_errors=True)
try:
    import tensorflowjs as tfjs
    tfjs.converters.save_keras_model(model, 'web_model')
except Exception as e:
    print('API Python indisponible, conversion via la ligne de commande...', str(e)[:100])
    !tensorflowjs_converter --input_format=keras koobo_model.h5 web_model
shutil.copy('classes.json', 'web_model/classes.json')
print('Fichiers générés :', os.listdir('web_model'))


## 11. Récupérer le modèle web
Crée `koobo_web_model.zip`. À décompresser dans **`web/public/model/`** de votre projet.
- **Kaggle** : le fichier apparaît dans l'onglet **Output** (panneau de droite) → bouton de téléchargement.
- **Colab** : le téléchargement démarre automatiquement.


In [ ]:
shutil.make_archive('koobo_web_model', 'zip', 'web_model')
print('✅ Archive créée : koobo_web_model.zip')
try:
    from google.colab import files   # Colab uniquement
    files.download('koobo_web_model.zip')
except Exception:
    print("Sur Kaggle : ouvrez l'onglet 'Output' à droite et téléchargez koobo_web_model.zip")


## 12. (Optionnel) Test rapide d'une prédiction


In [ ]:
import numpy as np
for imgs, labels in test.take(1):
    preds = model.predict(imgs)
    for i in range(3):
        print('Vrai :', CLASSES[labels[i].numpy()], '| Prédit :', CLASSES[int(np.argmax(preds[i]))],
              '(', round(float(np.max(preds[i])) * 100, 1), '%)')


---
### ➡️ Et après ?
1. Décompressez `koobo_web_model.zip` → mettez son contenu dans **`web/public/model/`**
   (vous devez avoir `web/public/model/model.json`, les `.bin` et `classes.json`).
2. Reconstruisez le frontend : `npm run build`.
3. La page **Détection → onglet « Hors-ligne »** chargera ce modèle et fera le diagnostic
   directement dans le navigateur, sans internet ni quota IA.
4. Re-téléversez `web/dist` sur cPanel. Terminé !
